# Model Notebook Template

**How to use this template:**

1. Copy this file and rename it to your notebook name, for example `05_decision_tree_random_forest.ipynb`.
2. In the **Model Settings** cell, change `MODEL_NAME`, the model and `param_grid`.
3. Use `make_scaled_pipeline(...)` for Logistic Regression, Naive Bayes, KNN and SVM. Tree-based models (Decision Tree, Random Forest, XGBoost) do not need scaling, so use the model directly.
4. For slow models (KNN and SVM), set `TUNING_SAMPLE_SIZE = 30000`. Tuning and cross-validation then use a smaller sample, but the final model is still trained on the full training set.
5. Run all cells. The results are saved to `results/metrics/` and the confusion matrices to `results/confusion_matrices/`.
6. If your notebook has two models, repeat sections 1 to 7 for the second model.
7. **Isolation Forest only:** it does not use labels and does not support `GridSearchCV` with F1 in the normal way. Train it on `X_train`, change its output with `(model.predict(X) == -1).astype(int)` so that 1 = Phishing, use `-model.score_samples(X)` as the score, and pass both to `evaluate_predictions(...)` from `src/evaluation.py`.

This example uses Logistic Regression. Delete this cell when you make your own notebook.

# 0X - Model Name

**Project:** DetectIt – Intelligent Web Threat Detection Using Machine Learning

**Goal:** Train and evaluate the model on the processed PhiUSIIL data, compare different feature sets, and test the URL-only version on the external dataset.

**Input:** `data/processed/`

**Output:** `results/metrics/`, `results/confusion_matrices/`

## Setup

We load the libraries and the shared functions from `src/`. Every team member uses the same functions, so all results are calculated in the same way and can be compared fairly.

In [ ]:
import sys
import time
from pathlib import Path

import pandas as pd
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

sys.path.append(str(Path("..").resolve()))
from src.data import get_feature_sets, load_train_test, load_url_experiment, sample_rows
from src.evaluation import (
    RANDOM_STATE,
    cross_validate_model,
    evaluate_model,
    make_cv,
    make_scaled_pipeline,
    plot_confusion_matrix,
    save_results,
)

In [ ]:
X_train, X_test, y_train, y_test = load_train_test()
feature_sets = get_feature_sets()

print(f"Training set: {X_train.shape[0]:,} rows | {X_train.shape[1]} features")
print(f"Test set:     {X_test.shape[0]:,} rows | {X_test.shape[1]} features")

## Model Settings

We choose the model, the settings we want to try during tuning, and the sample size for tuning.

In [ ]:
MODEL_NAME = "Logistic Regression"
model = make_scaled_pipeline(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))

param_grid = {
    "logisticregression__C": [0.01, 0.1, 1, 10],
}

TUNING_SAMPLE_SIZE = None

## 1. Hyperparameter Tuning

We try each combination of settings in `param_grid` using 3-fold cross-validation on the training set, and keep the one with the best F1-score. The test set is not used here.

In [ ]:
X_tune, y_tune = sample_rows(X_train, y_train, TUNING_SAMPLE_SIZE)

search = GridSearchCV(model, param_grid, scoring="f1", cv=make_cv(3), n_jobs=-1)
search.fit(X_tune, y_tune)
best_model = search.best_estimator_

print(f"Best settings: {search.best_params_}")
print(f"Best cross-validation F1: {search.best_score_:.4f}")

## 2. Cross-Validation

We check the best model with 5-fold stratified cross-validation. The mean shows the usual score, and the standard deviation (Std) shows how stable the model is. A small Std means the model gives similar results on different parts of the data.

In [ ]:
cv_results = cross_validate_model(best_model, X_tune, y_tune)
cv_results.round(4)

## 3. Final Test

We train the best model on the full training set and test it once on the test set. This is the score we report for this model.

In [ ]:
start = time.perf_counter()
final_model = clone(best_model).fit(X_train, y_train)
train_time = time.perf_counter() - start

test_results = evaluate_model(final_model, X_test, y_test)
print(f"Training time: {train_time:.1f} seconds")
pd.Series(test_results).round(4).to_frame("Test Score")

In [ ]:
plot_confusion_matrix(y_test, final_model.predict(X_test), f"{MODEL_NAME} - Test Set")

## 4. Feature Set Comparison

We train the same model on two smaller feature sets to answer two questions:

- **Original features only:** do our new features from feature engineering improve the model?
- **Without `URLSimilarityIndex`:** does the model still work well without its strongest feature?

In [ ]:
feature_results = {"All features": test_results}

for set_name in ["Original features only", "Without URLSimilarityIndex"]:
    features = feature_sets[set_name]
    set_model = clone(best_model).fit(X_train[features], y_train)
    feature_results[set_name] = evaluate_model(set_model, X_test[features], y_test)

pd.DataFrame(feature_results).T.round(4)

## 5. Experiment 2: URL Features Only

Here the model only uses the 18 URL features that we can create from any URL. We train it on PhiUSIIL and test it on both the PhiUSIIL test set and the external dataset. The difference between the two scores shows how well the model works on data from a different source.

In [ ]:
X_url_train, X_url_test, X_external, y_url_train, y_url_test, y_external = load_url_experiment()

url_model = clone(best_model).fit(X_url_train, y_url_train)
url_results = {
    "PhiUSIIL test set": evaluate_model(url_model, X_url_test, y_url_test),
    "External dataset": evaluate_model(url_model, X_external, y_external),
}
pd.DataFrame(url_results).T.round(4)

In [ ]:
plot_confusion_matrix(y_external, url_model.predict(X_external), f"{MODEL_NAME} - External Dataset")

## 6. Save the Results

We save all results in one table. The comparison notebook will later load the tables of all models.

In [ ]:
results = [
    {"Experiment": "Cross-validation", **cv_results["Mean"].to_dict()},
    {"Experiment": "Test set - All features", **test_results, "Train Time (s)": train_time},
    {"Experiment": "Test set - Original features only", **feature_results["Original features only"]},
    {"Experiment": "Test set - Without URLSimilarityIndex", **feature_results["Without URLSimilarityIndex"]},
    {"Experiment": "URL features - PhiUSIIL test set", **url_results["PhiUSIIL test set"]},
    {"Experiment": "URL features - External dataset", **url_results["External dataset"]},
]

save_results(MODEL_NAME, results).round(4)

## 7. Findings

Write the findings for this model here after running the notebook. For example:

- How good is the test score, and how stable is the cross-validation score?
- Did the new features help?
- How much did the score change without `URLSimilarityIndex`?
- How much did the score drop on the external dataset, and why?